# SSLP DBC/iDBC Notebook

Use this notebook to run minimal SSLP experiments with either `DBC` or `iDBC`.


In [1]:
using Pkg

function find_repo_root(start::AbstractString = pwd())
    dir = abspath(start)
    while true
        has_project = isfile(joinpath(dir, "Project.toml"))
        has_entry = isfile(joinpath(dir, "experiments", "sslp", "run_experiment.jl"))
        if has_project && has_entry
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repository root from $(start).")
        dir = parent
    end
end

repo_root = find_repo_root()
cd(repo_root)
Pkg.activate(repo_root)
println("repo_root = ", repo_root)


  Activating 

repo_root = /Users/aaron/StoCutDyProg

project at `~/StoCutDyProg`


## Edit Parameters

`cut_mode` can be `:DBC` or `:iDBC`.

For `:iDBC`, keep `mdc_iter >= 10` because the current SSLP run-grid skips `(iDBC, 5)`.


In [2]:
run_script = joinpath("experiments", "sslp", "run_experiment.jl")
config_path = joinpath("notebooks", "tmp_sslp_notebook_config.jl")

cut_mode = :SMC                     # :DBC or :iDBC
enable_copy_branching = (cut_mode == :iDBC)
enable_surrogate_copy_split = true
idbc_warm_pass = false

J = 5
I = 15
omega = 100
mdc_iter = 10

num_workers = 5
dry_run = false
max_iter = 10
terminate_time = 300.0
time_limit = 50.0
experiment_tag = "nb_sslp_$(cut_mode)"

config_text = """
EXPERIMENT_CONFIG = SslpExperimentConfig(
    num_workers = $(num_workers),
    dry_run = $(dry_run),
    static = SslpStaticConfig(
        max_iter = $(max_iter),
        terminate_time = $(terminate_time),
        time_limit = $(time_limit),
        enable_copy_branching = $(enable_copy_branching),
        enable_surrogate_copy_split = $(enable_surrogate_copy_split),
        copy_split_strategy = :surrogate_delta,
        idbc_warm_pass = $(idbc_warm_pass),
        experiment_tag = $(repr(experiment_tag)),
        legacy_logger_paths = false,
    ),
    sweep = SslpSweepConfig(
        omegas = Int[$(omega)],
        problem_sizes = Tuple{Int, Int}[($(J), $(I))],
        base_cuts = Symbol[],
        mdc_iters = Int[$(mdc_iter)],
        mdc_cuts = Symbol[$(repr(cut_mode))],
        normalization_candidates = Symbol[],
        normalization_cut = $(repr(cut_mode)),
        normalization_mdc_iter = $(mdc_iter),
    ),
)
"""

write(joinpath(repo_root, config_path), config_text)
println("Wrote config: ", joinpath(repo_root, config_path))
println(config_text)


Wrote config: /Users/aaron/StoCutDyProg/notebooks/tmp_sslp_notebook_config.jl
EXPERIMENT_CONFIG = SslpExperimentConfig(
    num_workers = 5,
    dry_run = false,
    static = SslpStaticConfig(
        max_iter = 10,
        terminate_time = 300.0,
        time_limit = 50.0,
        enable_copy_branching = false,
        enable_surrogate_copy_split = true,
        copy_split_strategy = :surrogate_delta,
        idbc_warm_pass = false,
        experiment_tag = "nb_sslp_SMC",
        legacy_logger_paths = false,
    ),
    sweep = SslpSweepConfig(
        omegas = Int[100],
        problem_sizes = Tuple{Int, Int}[(5, 15)],
        base_cuts = Symbol[],
        mdc_iters = Int[10],
        mdc_cuts = Symbol[:SMC],
        normalization_candidates = Symbol[],
        normalization_cut = :SMC,
        normalization_mdc_iter = 10,
    ),
)



In [3]:
cd(repo_root) do
    run(`julia --project=. $run_script $config_path`)
end


  Activating project at `~/StoCutDyProg`


Loaded config: /Users/aaron/StoCutDyProg/notebooks/tmp_sslp_notebook_config.jl
Total run cases: 1
Dry run mode: false
[SSLP] cut=SMC J=5 I=15 Ω=100 disj_iter=10 norm=SNC
------------------------------------------ Iteration Info ------------------------------------------------
Iter |        LB        |        UB        |       Gap      |      i-time     |    #D.     |     T-Time
----------------------------------------------------------------------------------------------------------
   1 |     -4000.00     |    150602.44     |   3865.06%     |      9.07 s     |     17     |      12.06 s     
   2 |     -3898.22     |      -611.11     |     84.32%     |      0.80 s     |      9     |      13.99 s     
   3 |      -676.90     |      -611.11     |      9.72%     |      0.55 s     |     10     |      14.54 s     
   4 |      -633.04     |      -611.11     |      3.47%     |      0.51 s     |      9     |      15.05 s     
   5 |      -628.40     |      -611.11     |      2.75%     |      0

Process(`julia --project=. experiments/sslp/run_experiment.jl notebooks/tmp_sslp_notebook_config.jl`, ProcessExited(0))

In [4]:
cd(repo_root) do
    run(`julia --project=. $run_script $config_path`)
end


  Activating project at `~/StoCutDyProg`


Loaded config: /Users/aaron/StoCutDyProg/notebooks/tmp_sslp_notebook_config.jl
Total run cases: 1
Dry run mode: false
[SSLP] cut=SMC J=5 I=15 Ω=100 disj_iter=10 norm=SNC
------------------------------------------ Iteration Info ------------------------------------------------
Iter |        LB        |        UB        |       Gap      |      i-time     |    #D.     |     T-Time
----------------------------------------------------------------------------------------------------------
   1 |     -4000.00     |    150602.44     |   3865.06%     |      8.91 s     |     17     |      10.54 s     
   2 |     -3898.22     |      -611.11     |     84.32%     |      0.79 s     |      9     |      12.52 s     
   3 |      -676.90     |      -611.11     |      9.72%     |      0.53 s     |     10     |      13.05 s     
   4 |      -633.04     |      -611.11     |      3.47%     |      0.51 s     |      9     |      13.58 s     
   5 |      -628.40     |      -611.11     |      2.75%     |      0

Process(`julia --project=. experiments/sslp/run_experiment.jl notebooks/tmp_sslp_notebook_config.jl`, ProcessExited(0))